In [16]:
# COCO 80 类名（YOLOv5 / YOLOv8 通用，供后续单元格共享使用）
NAMES = [
    'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light',
    'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
    'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee',
    'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard',
    'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple',
    'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch',
    'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard',
    'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase',
    'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]

In [7]:
import onnxruntime as ort
import numpy as np
import cv2

ONNX_PATH = r"d:\project\step1\week13\onnx_models\yolov5su.onnx"
IMGSZ = 640
CONF_THRES = 0.5
IOU_THRES = 0.45

class Yolov5_Infer:
    def __init__(self, onnx_path=r"d:\project\step1\week13\onnx_models\yolov5su.onnx"):
        self.session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
        self.input_name = self.session.get_inputs()[0].name
        self.output_name = self.session.get_outputs()[0].name
        self.ratio = 1

    def pre_process(self, frame_data):
        frame_data = cv2.cvtColor(frame_data, cv2.COLOR_BGR2RGB)
        h, w, c = frame_data.shape
        max_slide = max(h, w)
        img_zero = np.zeros((max_slide, max_slide, 3), dtype=np.uint8)
        img_zero[:h, :w] = frame_data
        self.ratio = max_slide / 640
        img0_re = cv2.resize(img_zero, (640, 640)) / 255
        img0_re = img0_re.astype(np.float32)
        img0_re = np.transpose(img0_re, axes=(2, 0, 1))
        img0_re = np.expand_dims(img0_re, axis=0)
        return img0_re

    def inference(self, image):
        return self.session.run([self.output_name], {self.input_name: image})

    def post_process(self, results):
        result = results[0][0].transpose(1, 0)
        print("输出形状:", result.shape)
        bboxes = []
        for bbox in result:
            cx, cy, w, h = bbox[:4]
            conf = float(bbox[4:].max())
            if conf > CONF_THRES:
                cls_idx = int(bbox[4:].argmax())
                top_x = int((cx - w / 2) * self.ratio)
                top_y = int((cy - h / 2) * self.ratio)
                origin_w = int(w * self.ratio)
                origin_h = int(h * self.ratio)
                bboxes.append([top_x, top_y, origin_w, origin_h, conf, cls_idx])

        if not bboxes:
            return np.array([])

        np_bboxes = np.array(bboxes)
        idx = cv2.dnn.NMSBoxes(np_bboxes[:, :4].tolist(),
                               np_bboxes[:, 4].tolist(),
                               CONF_THRES, IOU_THRES)
        idx = np.array(idx).ravel() if len(idx) else []
        return np_bboxes[idx]

    def show_img(self, bbx, img):
        for bb in bbx:
            top_x, top_y, origin_w, origin_h, conf, cls_idx = bb
            bottom_x, bottom_y = top_x + origin_w, top_y + origin_h
            cv2.rectangle(img, (int(top_x), int(top_y)),
                          (int(bottom_x), int(bottom_y)), (0, 0, 255), 1)
        cv2.imwrite("img.jpg", img)

    def forward(self, frame):
        img0 = self.pre_process(frame)
        results = self.inference(img0)
        filuter_result = self.post_process(results)
        self.show_img(filuter_result, frame)
        print(filuter_result)

img_data = cv2.imread(r"d:\project\step1\week13\action_train\01_left\raw_12.jpg")
yolov5_infer = Yolov5_Infer()
yolov5_infer.forward(img_data)

输出形状: (8400, 84)
[[174.           3.         465.         472.           0.90493846
    0.        ]]


In [17]:
import onnxruntime as ort
import numpy as np
import cv2

class Yolov8_Infer:
    def __init__(self, onnx_path=r"d:\project\step1\week13\onnx_models\yolov8n.onnx",
                 labels=None, conf_thres=0.25):
        self.session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
        self.input_name = self.session.get_inputs()[0].name
        self.output_name = self.session.get_outputs()[0].name
        self.labels = labels if labels is not None else NAMES
        self.colors = [self._color(i) for i in range(len(self.labels))]
        self.conf_thres = conf_thres
        self.ratio = 1

    def _color(self, i):
        palette = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0),
                   (0, 255, 255), (255, 0, 255), (128, 255, 0), (255, 128, 0),
                   (0, 128, 255), (255, 0, 128)]
        return palette[i % len(palette)]

    def pre_process(self, frame_data):
        frame_data = cv2.cvtColor(frame_data, cv2.COLOR_BGR2RGB)
        h, w, c = frame_data.shape
        max_slide = max(h, w)
        img_zero = np.zeros((max_slide, max_slide, 3), dtype=np.uint8)
        img_zero[:h, :w] = frame_data
        self.ratio = max_slide / 640
        img0_re = cv2.resize(img_zero, (640, 640)) / 255
        img0_re = img0_re.astype(np.float32)
        img0_re = np.transpose(img0_re, axes=(2, 0, 1))
        img0_re = np.expand_dims(img0_re, axis=0)
        return img0_re

    def inference(self, image):
        return self.session.run([self.output_name], {self.input_name: image})

    def post_process(self, results):
        result = results[0].transpose(0, 2, 1)[0]
        bboxes = []
        for bbox in result:
            cx, cy, w, h = bbox[:4]
            cls_scores = bbox[4:]
            cls_id = int(np.argmax(cls_scores))
            conf = float(cls_scores[cls_id])
            if conf > self.conf_thres:
                top_x = int((cx - w / 2) * self.ratio)
                top_y = int((cy - h / 2) * self.ratio)
                origin_w = int(w * self.ratio)
                origin_h = int(h * self.ratio)
                bboxes.append([top_x, top_y, origin_w, origin_h, conf, cls_id])

        if not bboxes:
            return np.array([])

        np_bboxes = np.array(bboxes)
        idx = cv2.dnn.NMSBoxes(np_bboxes[:, :4].tolist(),
                               np_bboxes[:, 4].tolist(), 0.5, 0.45)
        idx = np.array(idx).ravel() if len(idx) else []
        return np_bboxes[idx]

    def show_img(self, bbx, img):
        for bb in bbx:
            top_x, top_y, origin_w, origin_h, conf, cls_idx = bb
            bottom_x, bottom_y = top_x + origin_w, top_y + origin_h
            color = self.colors[int(cls_idx)]
            cv2.rectangle(img, (int(top_x), int(top_y)),
                          (int(bottom_x), int(bottom_y)), color, 2)
            cv2.putText(img, f"{self.labels[int(cls_idx)]} {conf:.2f}",
                        (int(top_x), int(top_y) - 10), 1, 0.8, color, 1)
        cv2.imwrite("v8_out.jpg", img)

    def forward(self, frame, show=True):
        img0 = self.pre_process(frame)
        results = self.inference(img0)
        dets = self.post_process(results)
        if show:
            self.show_img(dets, frame)
        return dets
    
    
img_data = cv2.imread(r"d:\project\step1\week13\action_train\01_left\raw_12.jpg")
yolov8_infer = Yolov8_Infer()
dets = yolov8_infer.forward(img_data)
print(f"检测到 {len(dets)} 个目标：")
for d in dets:
    top_x, top_y, w, h, conf, cls_id = d
    print(f"  {yolov8_infer.labels[int(cls_id)]:>12}  conf={conf:.2f}  "
          f"box=[{top_x},{top_y},{top_x+w},{top_y+h}]")    

检测到 2 个目标：
        person  conf=0.94  box=[177.0,1.0,639.0,476.0]
         couch  conf=0.62  box=[0.0,373.0,77.0,475.0]


In [ ]:
import onnxruntime
import numpy as np
import cv2 as cv2

class Yolov8_Pose_Infer:
    def __init__(self, model_path=r"d:\project\step1\week13\onnx_models\yolov8n-pose.onnx"):
        self.input_w = 640
        self.input_h = 640
        self.nms_threshold = 0.25
        self.conf_threshold = 0.5
        self.skeleton = [(0,1),(0,2),(1,3),(2,4),(5,6),(5,7),(7,9),(6,8),(8,10),(5,11),(6,12),
            (11,13),(13,15),(12,14),(14,16),(11,12),(5,6)]
        self.kpt_colors = [
            (0, 255, 0), (0, 255, 255), (0, 255, 255), (0, 255, 255), (0, 255, 255),
            (255, 0, 0), (255, 0, 0), (255, 0, 255), (255, 0, 255),
            (0, 0, 255), (0, 0, 255),
            (0, 255, 128), (0, 255, 128), (0, 255, 128), (0, 255, 128),
            (128, 0, 255), (128, 0, 255),
        ]
        self.session = onnxruntime.InferenceSession(model_path, providers=['CPUExecutionProvider'])
        
    def pre_process(self, frame):
        row, col, _ =frame.shape
        _max = max(col, row)
        result = np.zeros((_max, _max, 3),np.uint8)
        result[0:row, o:col] = frame
        return result
    
    def inference(self, frame):
        inputImage = self.pre_process(frame)
        scale_inputImage = cv2.resize(inputImage,dsize = (640,640))
        rgb = cv2.cvtColor(scale_inputImage, cv2.COLOR_BGR2RGB)
        rgb = np.transpose(rgb, (2, 0, 1))
        x_input = np.expand_dims(rgb, 0) / 255
        x_input = x_input.astype(np.float32)
        
        
        
    